# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² colorectal cancer dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible via the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

This package includes detailed clinicopathological and molecular data for 77 cancer survivors with second primary colorectal cancer, suitable for biomarker and anatomical distribution analysis. All resources, fields, and columns are referenced by their `@id`.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We load the dataset metadata and records using `mlcroissant`. This provides access to the dataset structure, record sets, and all schema information for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('Version:', metadata.version)
print('Date Published:', metadata.datePublished)
print('Identifier:', metadata.identifier)
print('Dataset @id:', metadata.id)
if hasattr(metadata, 'keywords'):
    print('Keywords:', metadata.keywords)
if hasattr(metadata, 'personalSensitiveInformation'):
    print('Personal Sensitive Information:', metadata.personalSensitiveInformation)

## 2. Data Overview

Explore available record sets (tables), their fields and columns. All are referenced using their Croissant `@id` to ensure unambiguous usage.

**Note:** Record set, field, and column `@id`s can be accessed via `dataset.record_sets`, `record_set.fields`, and `field.columns` respectively.

In [ ]:
# List all record sets with their ids and names
record_sets = dataset.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f'  @id: {rs.id}   |   name: {getattr(rs, "name", "N/A")}')

# For each record set, print fields and columns (by @id)
for rs in record_sets:
    print(f'\nRecord Set: {rs.id}')
    if hasattr(rs, 'fields'):
        print('  Fields:')
        for f in rs.fields:
            print(f'    Field @id: {f.id}   |   name: {getattr(f, "name", "N/A")}')
            if hasattr(f, 'columns'):
                print('      Columns:')
                for c in f.columns:
                    print(f'        Column @id: {c.id}   |   name: {getattr(c, "name", "N/A")}')

## 3. Data Extraction

We extract records from each main record set using their Croissant `@id`. The resulting pandas DataFrames are indexed by each record set's `@id` for easy access.

Let's load records from each table and view their column names and sample data.

In [ ]:
# Prepare to load all record sets into DataFrames, referenced by @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

for record_set_id in record_set_ids:
    print(f'\nColumns for record set {record_set_id}:')
    print(dataframes[record_set_id].columns.tolist())
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a main record set (table) for typical clinical variable analysis.

We will:
- Choose a numeric field (such as Age at Diagnosis) using its field `@id` and explore/filter it.
- Normalize the numeric variable.
- Group by an attribute (e.g., Sex or Anatomical site) referenced by its field `@id`.

_**Note**: Be sure to update the field `@id`s below using the code output in the previous section for this dataset if you explore further._

In [ ]:
# === Update these variables to match your dataset's @id ===
# For this example, let's assume the main record set has @id:
# 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#ClinicalRecordSet'
# And fields for age and sex, example @id:
#   'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#AgeAtSecondDiagnosis' (numeric)
#   'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#Sex' (categorical group)

main_record_set_id = None
age_field_id = None
sex_field_id = None

# Attempt to infer ids based on available dataframes
for rs in record_sets:
    # Heuristic: select the largest table as the main clinical record set
    df = dataframes[rs.id]
    if df.shape[0] == 77:
        main_record_set_id = rs.id
        possible_age_fields = [col for col in df.columns if 'age' in col.lower()]
        possible_sex_fields = [col for col in df.columns if 'sex' in col.lower() or 'gender' in col.lower()]
        if possible_age_fields:
            age_field_id = possible_age_fields[0]
        if possible_sex_fields:
            sex_field_id = possible_sex_fields[0]
        break

if main_record_set_id is None:
    # Fallback: select the first, warn the user
    print('Warning: Main record set not found, using the first available.')
    main_record_set_id = record_set_ids[0]

df = dataframes[main_record_set_id]

# Display columns for mapping
print('Main record set id:', main_record_set_id)
print('Available columns:', df.columns.tolist())
print('Selected numeric (age) field id:', age_field_id)
print('Selected group (sex) field id:', sex_field_id)

# Continue analysis only if age_field_id is found and numeric
if age_field_id is not None and pd.api.types.is_numeric_dtype(df[age_field_id]):
    threshold = 50  # Example: age > 50
    filtered_df = df[df[age_field_id] > threshold].copy()
    print(f'Filtered records with {age_field_id} > {threshold}:')
    display(filtered_df.head())

    # Normalize
    norm_col = f"{age_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std(ddof=0)

    print(f'Normalized {age_field_id}:')
    display(filtered_df[[age_field_id, norm_col]].head())

    # Group by sex if available
    if sex_field_id and sex_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(sex_field_id)[age_field_id].mean()
        print(f'Mean {age_field_id} by {sex_field_id}:')
        print(grouped)
else:
    print('No numeric age field found for EDA. Please update field ids.' )

## 5. Visualization

Let's visualize the distribution of the selected numeric variable (such as Age at Second Diagnosis), and, if available, plot by a group such as Sex.

_If needed, install matplotlib and seaborn._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if age_field_id is not None and age_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[age_field_id].dropna(), bins=10, kde=True)
    plt.title(f'{age_field_id} Distribution')
    plt.xlabel('Age')
    plt.ylabel('Count')
    plt.show()

    if sex_field_id and sex_field_id in df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=sex_field_id, y=age_field_id, data=df)
        plt.title(f'{age_field_id} by {sex_field_id}')
        plt.show()
else:
    print('Age field not available for visualization.')

## 6. Conclusion

* In this notebook, we demonstrated loading the FAIR² colorectal cancer dataset directly via its Croissant schema using the `mlcroissant` library, referencing all schema elements by their `@id` fields for reproducibility and clarity.
* We explored dataset metadata, listed all record sets, fields, and columns, and extracted records for further analysis.
* Using an exemplary numeric variable (such as age at diagnosis), we filtered, normalized, and grouped the data, then visualized distributions and group comparisons.

**Tips for further analysis:**
- Use additional record sets and fields by referencing their `@id`.
- Apply advanced analytics and modeling as needed for your research question.
- For reproducible pipelines, always refer to Croissant elements by their `@id`.

For more, see [mlcroissant documentation](https://github.com/mlcommons/croissant) and the [FAIR² Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).